# Does the cell-line name join pair the right two lines?

The production join matches SCP542's `Cell_line` (the part before the first `_`) against CTRPv2's
`ccl_name`, both lowercased with hyphens stripped (`_normalize_cell_line`,
`scripts/preprocessing/ctrp_to_h5ad.py`). A normalised name is not an identifier: two different cell
lines can carry names that collapse onto the same key, and the join would silently pair them. Until
now the only check was that the two sides agreed on CCLE primary site.

**What changed 11.08.2026.** The response data moved to DrEval's reprocessed CTRPv2, which ships a
**Cellosaurus accession on every row** and a **Cellosaurus release alongside it**. That makes an
independent check possible: resolve SCP542's own names against Cellosaurus, and compare the accession
obtained that way against the accession CTRP ships for the line the join paired it with. A
disagreement means the key matched two distinct cell lines.

**What is decided (11.08.2026, Selin).**

| | |
|---|---|
| Join key | **Unchanged** — the normalised name stays the production key. Accessions were measured as an alternative and are *worse*: they resolve 172 of 198 SCP542 lines against the name join's 180, because Cellosaurus names collide more often than CTRP's do |
| `cellosaurus_id` | **Carried onto the target as a recorded attribute**, not used as a key. It gives every cell line a persistent identifier for citation and for the planned cross-database step, without the join depending on it |
| `h292` → `ncih292` | **Alias retained.** Cellosaurus independently maps both spellings to `CVCL_0455`, so the alias is now corroborated rather than merely evidenced |
| Ambiguity | **Resolved by three stated rules, never by picking a winner per case** — see below. A name that survived all three rules with more than one candidate would be reported unresolved, not guessed |

**Sources.** Cellosaurus **release 52.0, 10 April 2025** (CC BY 4.0) and the CTRPv2 response data, both
from Zenodo record [`21807175`](https://doi.org/10.5281/zenodo.21807175) (*Dataset for drevalpy*,
published 2026-08-05). Fetched, MD5-verified and cached by
`scripts/sources/fetch_ctrp_response.py`, which writes the record, DOI and retrieval date into
`provenance.json` beside the data.

**Everything below runs through `scripts/sources/cellosaurus.py`** — the parser and the
resolution rules live there, not in this notebook, so the pipeline and this verification cannot drift
apart. The notebook only reports.

## The three ambiguity rules

Ten of SCP542's 198 names match more than one Cellosaurus entry (nine of them among the 180 that join
to CTRPv2). Each rule below removes candidates on a property Cellosaurus records, so the outcome does
not depend on anyone's judgement about which line was *meant*.

They are **tie-breakers, not filters**: a rule is consulted only while more than one candidate remains.
A name matching exactly one entry keeps it even if a rule would have rejected that entry — so a
wrong-species match surfaces as a mismatch instead of silently vanishing.

| Rule | Why it is not a judgement call | Resolves |
|---|---|---|
| **1. Human only** (`OX` = *Homo sapiens*) | SCP542 is a human cancer cell-line panel; a non-human entry cannot be the line that was sequenced | `TE1`, `A204`, `ABC1`, `HT55` — the competitor is a **mouse** entry in each case |
| **2. Cancer cell lines only** (`CA` = *Cancer cell line*) | Same reason: the panel contains no stem-cell or iPSC lines | `C32` (competitors are two induced-pluripotent lines), `RCM1` (competitor is a human ESC line) |
| **3. A primary identifier beats a synonym** | Cellosaurus' `ID` is the entry's own name; an `SY` hit means some *other* entry happens to list this string among its alternatives | `PC3` (7 candidates; only `CVCL_0035` holds it as its own name, `PC-3`), `EBC1` (vs `NB-EbC1`), `SCC25` (vs `UM-SCC-25`), `SCC9` (vs `UM-SCC-9`) |

Applied in that order, **all ten resolve to exactly one accession and none is left ambiguous**, and
every one of the nine that joins agrees with the accession CTRP ships — so the rules are corroborated
externally rather than merely being self-consistent.

Two names resolve to nothing at all: `SCC47` and `93VU` are absent from Cellosaurus 52.0. Neither has a
CTRPv2 row either, so nothing is lost.

A first, cruder pass without these rules fell through to a synonym and returned *HNC PC3* (head and
neck) for `PC3` and a human ESC line for `RCM1`. Recorded because it shows what the rules are for.

In [1]:
from pathlib import Path

import anndata as ad
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.layout import DEFAULT_CTRP_SCORE, PipelinePaths

# Derived, not hardcoded (13.08.2026, #25): both were absolute /Users paths. DREVAL resolves
# to the identical directory -- the record number comes from layout.ZENODO_RESPONSE_RECORD,
# so it can no longer drift from the pinned record.
#
# ** TARGETS CHANGES WHICH FILE IS READ, and that is Selin's call, not a tidy-up. ** The
# literal ended in _auc.h5ad -- the target RETIRED on 11.08.2026 as defectively normalised.
# Deriving resolves to the current default, 'auc_cc'. Spelled out rather than left implicit.
_paths = PipelinePaths.build(None, 'hvg5000', DEFAULT_CTRP_SCORE)
DREVAL = _paths.drevalpy_dir
TARGETS = _paths.targets_h5ad
print(f'target artifact: {TARGETS.name}')

# The production join key, verbatim from ctrp_to_h5ad._normalize_cell_line. Anything looser would be
# testing a different join than the one that runs.
def prod_key(s):
    return str(s).strip().lower().replace('-', '')


# Cellosaurus writes names with spaces and dots that our key never has to handle, because it only ever
# sees SCP542's and CTRP's spellings. Used ONLY to look names up in Cellosaurus, never as a join key.
def cello_key(s):
    return prod_key(s).replace(' ', '').replace('.', '')


# SCP542 labels are like MDAMB361_BREAST: the line, then the tissue. Both halves are used -- the name
# for the join, the tissue as an independent check that does not go through Cellosaurus at all.
obs = ad.read_h5ad(TARGETS, backed='r').obs
scp = (pd.DataFrame({'scp_label': obs['Cell_line'].astype(str).unique()})
       .assign(scp_name=lambda d: d.scp_label.str.split('_').str[0],
               scp_tissue=lambda d: d.scp_label.str.split('_', n=1).str[1].fillna(''))
       .assign(key=lambda d: d.scp_name.map(prod_key)))

# One row per CTRP cell line: its name, the accession DrEval ships for it, and its tissue.
ctrp = (pd.read_csv(DREVAL / 'CTRPv2' / 'CTRPv2.csv', usecols=['cellosaurus_id', 'ccl_name'])
        .drop_duplicates('ccl_name')
        .merge(pd.read_csv(DREVAL / 'CTRPv2' / 'cell_line_names.csv'), on='cellosaurus_id', how='left')
        .assign(key=lambda d: d.ccl_name.map(prod_key)))

print(f'SCP542 cell lines : {len(scp):,}')
print(f'CTRPv2 cell lines : {len(ctrp):,}')
print(f'distinct SCP542 names collapsing onto one key : '
      f'{int((scp.groupby("key").scp_name.nunique() > 1).sum())}')
print(f'distinct CTRP names collapsing onto one key   : '
      f'{int((ctrp.groupby("key").ccl_name.nunique() > 1).sum())}')

target artifact: SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad


SCP542 cell lines : 198
CTRPv2 cell lines : 886
distinct SCP542 names collapsing onto one key : 0
distinct CTRP names collapsing onto one key   : 0


## 1. Resolving both sides to an accession

`scripts/sources/cellosaurus.py` does the work: `load_cellosaurus` parses the flat file (and
reads the release straight out of its header, so the version cannot be misquoted), and
`resolve_accessions` applies the three tie-break rules, recording which one fired for each name.

The CTRP side needs no resolution — DrEval ship an accession on every row. Resolving SCP542's names
independently is what makes the comparison a *check* rather than a restatement.

In [2]:
import sys

sys.path.insert(0, str(ROOT))
from scripts.sources.cellosaurus import load_cellosaurus, resolve_accessions  # noqa: E402

cello, release = load_cellosaurus(DREVAL / 'meta' / 'cellosaurus.txt')
print(f'{release}: {cello.accession.nunique():,} entries, {len(cello):,} names '
      f'({int((cello.kind == "synonym").sum()):,} of them synonyms)')

resolved = resolve_accessions(scp.scp_name, cello)
print('\nstatus of the 198 SCP542 names:')
print(resolved.status.value_counts().to_string())

# Every name that needed a tie-break, and which rule settled it -- the table in the header above.
print('\nnames matching more than one entry:')
print(resolved[resolved.n_candidates > 1]
      [['name', 'n_candidates', 'accession', 'kind', 'resolved_by', 'status']]
      .sort_values(['resolved_by', 'name']).to_string(index=False))

print('\nnames absent from Cellosaurus entirely:',
      resolved.loc[resolved.status == 'absent', 'name'].tolist())

Cellosaurus 52.0 (10-April-2025): 163,868 entries, 288,394 names (124,526 of them synonyms)



status of the 198 SCP542 names:
status
resolved    196
absent        2

names matching more than one entry:
 name  n_candidates accession       kind                          resolved_by   status
 EBC1             2 CVCL_2891 identifier a primary identifier beats a synonym resolved
  PC3             7 CVCL_0035 identifier a primary identifier beats a synonym resolved
SCC25             2 CVCL_1682 identifier a primary identifier beats a synonym resolved
 SCC9             2 CVCL_1685 identifier a primary identifier beats a synonym resolved
  C32             3 CVCL_1097    synonym               cancer cell lines only resolved
 RCM1             2 CVCL_1648    synonym               cancer cell lines only resolved
 A204             2 CVCL_1058 identifier                           human only resolved
 ABC1             2 CVCL_1066 identifier                           human only resolved
 HT55             2 CVCL_1294 identifier                           human only resolved
  TE1             2 C

## 2. Does the join pair the right two lines?

For every SCP542 line the name join matches to a CTRP line, compare **the accession resolved from
SCP542's own name** against **the accession DrEval ship for the CTRP line it was matched to**. These
come from different places — one from Cellosaurus by name, one from DrEval's own curation — so a
disagreement means the normalised key matched two distinct cell lines.

A second, independent check follows that does not go through Cellosaurus at all: SCP542 labels carry
the tissue in the suffix (`MDAMB361_BREAST`), and DrEval ship a tissue per line. Two lines that are
really the same line cannot sit in different tissues.

In [3]:
joined = (scp.merge(ctrp, on='key', how='inner')
          .merge(resolved[['name', 'accession', 'status']],
                 left_on='scp_name', right_on='name', how='left'))
print(f'SCP542 lines matched to a CTRP line by name: {len(joined)} of {len(scp)}')

agree = joined.accession == joined.cellosaurus_id
unresolvable = joined.accession.isna()
print(f'  accessions agree      : {int((agree & ~unresolvable).sum())}')
print(f'  accessions DISAGREE   : {int((~agree & ~unresolvable).sum())}')
print(f'  SCP542 name unresolved: {int(unresolvable.sum())}')
if int((~agree & ~unresolvable).sum()):
    print(joined[~agree & ~unresolvable]
          [['scp_name', 'accession', 'ccl_name', 'cellosaurus_id', 'tissue']].to_string(index=False))

# --- independent of Cellosaurus: SCP542's tissue suffix vs DrEval's tissue ---
# The two vocabularies differ; these pairs are the same tissue under different names, checked by hand
# against the lines they affect (bladder lines filed under urinary tract, cholangiocarcinoma lines
# under liver, rhabdomyosarcoma lines under muscle).
TISSUE_SYNONYMS = {
    'URINARYTRACT': 'BLADDER', 'BILIARYTRACT': 'LIVER', 'SOFTTISSUE': 'MUSCLE',
    'LARGEINTESTINE': 'COLON', 'AUTONOMICGANGLIA': 'NERVOUSSYSTEM', 'PLEURA': 'LUNG',
    'HAEMATOPOIETICANDLYMPHOID': 'BLOOD', 'UPPERAERODIGESTIVETRACT': 'HEADANDNECK',
    'CENTRALNERVOUSSYSTEM': 'BRAIN', 'OESOPHAGUS': 'ESOPHAGUS', 'ENDOMETRIUM': 'UTERUS',
    'BONE': 'SARCOMA', 'BILIARYTRACT ': 'BILEDUCT',
}


def tissue_agrees(scp_tissue, dreval_tissue):
    a = str(scp_tissue).upper().replace('_', '').replace(' ', '')
    b = str(dreval_tissue).upper().replace('_', '').replace(' ', '')
    return a == b or a in b or b in a or TISSUE_SYNONYMS.get(a) == b


joined['tissue_ok'] = [tissue_agrees(a, b) for a, b in zip(joined.scp_tissue, joined.tissue)]
print(f'\ntissue check over the same {len(joined)} matches:')
print(f'  agree                 : {int(joined.tissue_ok.sum())}')
print(f'  disagree              : {int((~joined.tissue_ok).sum())}')
if int((~joined.tissue_ok).sum()):
    print(joined[~joined.tissue_ok][['scp_name', 'scp_tissue', 'ccl_name', 'tissue']]
          .to_string(index=False))

SCP542 lines matched to a CTRP line by name: 180 of 198
  accessions agree      : 180
  accessions DISAGREE   : 0
  SCP542 name unresolved: 0

tissue check over the same 180 matches:
  agree                 : 180
  disagree              : 0


## 3. What the target build actually produces

The join is verified; this section records what comes out of it, by calling the pipeline's own
functions rather than reimplementing them — so these counts cannot drift from what the `targets` step
does. Nothing is written: `run()` is not called, only the loader, the de-duplication and the drug
filter.

**Why 181 here and 180 above.** Section 2 joins the two name sets raw. The pipeline additionally
applies `CTRP_CELL_LINE_ALIASES`, which maps CTRP's `H292` onto SCP542's `NCIH292` — one line CTRP
spells without the `NCI` prefix it uses for 106 others. Cellosaurus maps both spellings to
`CVCL_0455`, which is what turns that alias from a guess into an identification. See
[Step 01](../../../docs/steps/01-datasets-and-harmonization.md#the-join-dropped-a-screened-cell-line-h292-10082026).

**The two sources are not the same curve set.** DrEval re-fit from CTRPv2's raw dose-response
measurements under CurveCurator's own quality control, rather than taking CTRP's published post-QC
fits — so the sets of surviving (cell line, drug) pairs differ **in both directions**. The comparison
below quantifies that, because a one-directional difference would mean rows were lost and a symmetric
one means the QC simply disagrees.

In [4]:
import numpy as np  # noqa: E402

from scripts.preprocessing import ctrp_to_h5ad as C  # noqa: E402
from scripts.layout import CTRP_SCORES  # noqa: E402

# The pipeline's own key function, applied to the same names -- vectorised over the Series, which is
# how ctrp_to_h5ad calls it.
scp_keys = set(C._normalize_cell_line(scp.scp_name))
summary = []
for score in CTRP_SCORES:
    print(f'===== {score} ({C.SCORE_COLUMNS[score]}) =====')
    full = C._deduplicate_measurements(C._load_drevalpy_long(DREVAL / 'CTRPv2' / 'CTRPv2.csv', score))
    overlap = scp_keys & set(full.ccl_name_norm)
    long_ov, kept = C._build_drug_table(full, overlap_cell_lines_norm=overlap,
                                        min_cell_lines=50, target_drugs=None)
    Y = long_ov.pivot(index='ccl_name_norm', columns='cpd_name_norm',
                      values='score').reindex(columns=kept)
    v = Y.to_numpy()[~np.isnan(Y.to_numpy())]
    print(f'  overlap {len(overlap)} lines | matrix {Y.shape} | observed {int(Y.notna().sum().sum()):,} '
          f'| density {100 * Y.notna().mean().mean():.1f} %')
    print(f'  values: min {v.min():.3f}  median {np.median(v):.3f}  max {v.max():.3f}\n')
    summary.append({'measure': score, 'lines': Y.shape[0], 'drugs': Y.shape[1],
                    'observed': int(Y.notna().sum().sum()),
                    'density_%': round(100 * Y.notna().mean().mean(), 1),
                    'min': round(float(v.min()), 3), 'median': round(float(np.median(v)), 3),
                    'max': round(float(v.max()), 3)})

print(pd.DataFrame(summary).to_string(index=False))

===== auc_cc (AUC_curvecurator) =====


  395,024 measurements | 886 cell lines | 545 drugs
  8,187 of 395,024 rows are exact duplicates of another row (2.1 %) and are dropped.
  Drug filter: 534 / 545 drugs kept (>= 50 overlapping cell lines).
  overlap 181 lines | matrix (181, 534) | observed 81,906 | density 84.7 %
  values: min 0.020  median 0.925  max 1.830

===== ln_ic50_cc (LN_IC50_curvecurator) =====


  159,050 of 395,024 rows have no LN_IC50_curvecurator (40.3 %) and are dropped.
  235,974 measurements | 886 cell lines | 545 drugs
  4,346 of 235,974 rows are exact duplicates of another row (1.8 %) and are dropped.
  Drug filter: 365 / 539 drugs kept (>= 50 overlapping cell lines).
  overlap 181 lines | matrix (181, 365) | observed 42,584 | density 64.5 %
  values: min -11.385  median 2.487  max 8.634

   measure  lines  drugs  observed  density_%     min  median   max
    auc_cc    181    534     81906       84.7   0.020   0.925 1.830
ln_ic50_cc    181    365     42584       64.5 -11.385   2.487 8.634
